[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/altair-certified/notebooks/day-03-marks-deep-dive.ipynb#scrollTo=aa1b2c3d)

---
# Day 3 · Marks Deep Dive
**certified-journeys / altair-certified** · Day 3 · Practice

> **Goal for today:** Build 8 chart types (scatter with jitter, multi-series line, grouped/stacked bar, area, heatmap, pie/donut, text labels, rule reference line) and combine them into a layered chart.


In [ ]:
%pip install -q altair vega-datasets


In [ ]:
import altair as alt
import pandas as pd
from vega_datasets import data

# Load datasets we'll use throughout this notebook
cars = data.cars()
stocks = data.stocks()
movies = data.movies().dropna(subset=['Rotten_Tomatoes_Rating', 'IMDB_Rating', 'Major_Genre'])

print("cars:", cars.shape)
print("stocks:", stocks.shape)
print("movies:", movies.shape)


## Part 1 · `mark_point` — Scatter plot with jitter

A plain scatter plot stacks points on top of each other when many have the same
x-value (overplotting). **Jitter** adds random offsets so individual points separate.

In Altair, jitter is applied with `transform_calculate` + `jitter()` noise function:

```python
.transform_calculate(jitter='sqrt(-2*log(random()))*cos(2*PI*random())')
```

This uses the Box-Muller transform to generate normally-distributed random offsets.
Combine with `alt.X(field='jitter', type='quantitative')` and `scale=alt.Scale(zero=False)`.


In [ ]:
# Scatter plot with jitter to separate overlapping points by Origin
jitter_chart = alt.Chart(cars).mark_point(
    size=40,
    opacity=0.6
).transform_calculate(
    # Box-Muller normally-distributed jitter
    jitter='sqrt(-2*log(random()))*cos(2*PI*random())'
).encode(
    x=alt.X('jitter:Q',
             title=None,
             scale=alt.Scale(zero=False),
             axis=alt.Axis(labels=False, ticks=False, grid=False)),  # hide jitter axis
    y=alt.Y('Miles_per_Gallon:Q', title='Miles per Gallon'),
    color=alt.Color('Origin:N', title='Origin'),
    column=alt.Column('Origin:N', title=''),
    tooltip=['Name', 'Origin', 'Miles_per_Gallon', 'Horsepower']
).properties(
    title='MPG Distribution by Origin (jittered strip plot)',
    width=140,
    height=300
)

jitter_chart


### What just happened?
- `transform_calculate` adds a new computed column (`jitter`) without modifying the source DataFrame.
- We facet by `Origin` with `column=` — each column gets its own panel.
- `axis=alt.Axis(labels=False, ticks=False, grid=False)` hides the meaningless jitter x-axis.
- **`mark_point(size=40, opacity=0.6)`** — size and opacity are static mark properties, not encoded.
- The result reveals the distribution shape within each group — more useful than a box plot for small datasets.


## Part 2 · `mark_line` — Multi-series time series

Line charts need data sorted by x. Altair handles this automatically for temporal axes.
For multiple series, use `color` or `detail` encoding to split by group.

The `detail` encoding groups lines **without** applying color — useful when you want
a single color with multiple lines, or when you're already using color for something else.


In [ ]:
# Multi-series stock price line chart
# Filter to 3 companies for clarity
stocks_3 = stocks[stocks['symbol'].isin(['AAPL', 'GOOG', 'MSFT'])]

line_chart = alt.Chart(stocks_3).mark_line().encode(
    x=alt.X('date:T', title='Date'),
    y=alt.Y('price:Q', title='Stock Price (USD)', scale=alt.Scale(zero=False)),
    color=alt.Color('symbol:N', title='Company',
                    scale=alt.Scale(scheme='tableau10')),  # named color scheme
    tooltip=[
        alt.Tooltip('date:T', format='%Y-%m'),
        'symbol:N',
        alt.Tooltip('price:Q', format='$.2f')
    ]
).properties(
    title='Stock Prices Over Time — AAPL, GOOG, MSFT',
    width=560,
    height=280
)

line_chart


### What just happened?
- `color='symbol:N'` automatically groups the line by company and draws separate lines.
- `alt.Scale(scheme='tableau10')` applies a named Vega color palette — no manual hex colors needed.
- `alt.Tooltip('date:T', format='%Y-%m')` formats the date tooltip as `YYYY-MM`.
- `alt.Tooltip('price:Q', format='$.2f')` formats price with a dollar sign and 2 decimals.
- **`scale=alt.Scale(zero=False)`** on y prevents the chart from wasting space at 0 — stock prices are never 0.


## Part 3 · `mark_bar` — Grouped and stacked bar charts

| Bar type | How to achieve |
|----------|---------------|
| Simple bar | `mark_bar()` with one categorical x |
| Grouped bar | Add `xOffset=alt.XOffset('group:N')` |
| Stacked bar | Default when `color` encodes a categorical with multiple values per x |
| Normalized | `stack='normalize'` on y encoding |

Grouped and stacked bars both split one x-category into sub-categories — the
difference is whether sub-bars sit side-by-side (grouped) or on top of each other (stacked).


In [ ]:
# Grouped bar chart: mean Horsepower by Origin and Cylinders
cars_clean = cars.dropna(subset=['Horsepower', 'Cylinders'])
# Only keep cylinder counts that appear across multiple origins
cars_cyl = cars_clean[cars_clean['Cylinders'].isin([4, 6, 8])].copy()
cars_cyl['Cylinders'] = cars_cyl['Cylinders'].astype(str) + ' cyl'

grouped_bar = alt.Chart(cars_cyl).mark_bar().encode(
    x=alt.X('Origin:N', title='Country of Origin'),
    y=alt.Y('mean(Horsepower):Q', title='Mean Horsepower'),
    xOffset='Cylinders:N',           # groups bars side-by-side within each Origin
    color=alt.Color('Cylinders:N', title='Cylinders'),
    tooltip=['Origin:N', 'Cylinders:N', alt.Tooltip('mean(Horsepower):Q', format='.1f')]
).properties(
    title='Mean Horsepower by Origin and Cylinder Count (grouped bar)',
    width=360,
    height=280
)

grouped_bar


In [ ]:
# Stacked bar chart: count of cars by Origin and Cylinders
stacked_bar = alt.Chart(cars_cyl).mark_bar().encode(
    x=alt.X('Origin:N', title='Country of Origin'),
    y=alt.Y('count()', title='Number of Models'),
    color=alt.Color('Cylinders:N', title='Cylinders'),
    tooltip=['Origin:N', 'Cylinders:N', 'count()']
).properties(
    title='Car Models by Origin and Cylinder Count (stacked bar)',
    width=300,
    height=280
)

stacked_bar


### What just happened?
- **Grouped**: `xOffset='Cylinders:N'` shifts bars sideways within each x-group — bars share origin, stand side-by-side.
- **Stacked**: remove `xOffset` and Altair stacks the colored segments automatically.
- `mean(Horsepower):Q` is shorthand for `aggregate='mean', field='Horsepower'` — inline aggregation.
- **Rule of thumb**: grouped bars are better for comparing individual values; stacked bars show composition and totals.


## Part 4 · `mark_area` — Overlapping area chart with opacity

`mark_area` fills the region between the line and the baseline (default: 0).
For overlapping areas (multiple series), set `opacity` < 1 so series behind show through.

```python
mark_area(opacity=0.4)
```

For a stacked area chart (no overlap), just drop the opacity — Altair stacks by default.


In [ ]:
# Overlapping area chart: stock price history with opacity
stocks_2 = stocks[stocks['symbol'].isin(['AAPL', 'MSFT'])]

area_chart = alt.Chart(stocks_2).mark_area(
    opacity=0.45,   # static property — not data-driven
    line=True       # add outline line on top of area for clarity
).encode(
    x=alt.X('date:T', title='Date'),
    y=alt.Y('price:Q', title='Stock Price (USD)', stack=None),  # stack=None for overlap
    color=alt.Color('symbol:N', title='Company'),
    tooltip=['symbol:N', 'date:T', 'price:Q']
).properties(
    title='AAPL vs MSFT Stock Price (overlapping area, opacity=0.45)',
    width=540,
    height=260
)

area_chart


### What just happened?
- `mark_area(opacity=0.45)` fills under the line with semi-transparent color.
- `stack=None` on the y encoding tells Altair **not** to stack the areas — they overlap instead.
- `line=True` draws a solid line along the top of each area — makes the trend easier to read.
- **Without `stack=None`**: Altair would stack MSFT on top of AAPL, showing total, not comparison.
- Use overlapping areas for 2–3 series max; with more series, use a line chart to avoid confusion.


## Part 5 · `mark_rect` — Heatmap from aggregated data

A heatmap uses colored rectangles to show the intensity of a third variable
across two categorical dimensions. In Altair:

- x = one categorical variable
- y = another categorical variable  
- color = quantitative value (aggregated)

`mark_rect` fills each cell with a color mapped to the value.
Use `alt.Color(scale=alt.Scale(scheme='...'))` to choose a perceptually uniform scheme.


In [ ]:
# Heatmap: mean IMDB rating by Genre and decade
movies_hm = movies.copy()
movies_hm['Release_Year'] = pd.to_datetime(movies_hm['Release_Date'], errors='coerce').dt.year
movies_hm['Decade'] = (movies_hm['Release_Year'] // 10 * 10).astype('Int64').astype(str) + 's'
movies_hm = movies_hm.dropna(subset=['Decade', 'Major_Genre', 'IMDB_Rating'])
# Limit to top genres by count for a clean chart
top_genres = movies_hm['Major_Genre'].value_counts().head(8).index.tolist()
movies_hm = movies_hm[movies_hm['Major_Genre'].isin(top_genres)]

heatmap = alt.Chart(movies_hm).mark_rect().encode(
    x=alt.X('Decade:O', title='Decade', sort=['1970s', '1980s', '1990s', '2000s', '2010s']),
    y=alt.Y('Major_Genre:N', title='Genre', sort='-x'),
    color=alt.Color(
        'mean(IMDB_Rating):Q',
        title='Avg IMDB Rating',
        scale=alt.Scale(scheme='yellowgreenblue', domain=[5.5, 8.0])
    ),
    tooltip=[
        'Major_Genre:N', 'Decade:O',
        alt.Tooltip('mean(IMDB_Rating):Q', format='.2f', title='Avg IMDB'),
        alt.Tooltip('count()', title='# Movies')
    ]
).properties(
    title='Mean IMDB Rating by Genre and Decade',
    width=360,
    height=280
)

heatmap


### What just happened?
- `mark_rect()` fills each genre×decade cell with a color representing the mean IMDB rating.
- `alt.Scale(scheme='yellowgreenblue', domain=[5.5, 8.0])` pins the color scale to a meaningful range — without `domain`, it would span the full data range which compresses the perceptible differences.
- `sort='-x'` on the y-axis sorts genres by descending x-count — highest-volume genres at top.
- **Heatmaps are best** when you want to show a continuous value across two categorical dimensions simultaneously.


## Part 6 · `mark_arc` — Pie and donut chart

`mark_arc` was added in Altair 4.2 / Vega-Lite 5. It uses:

- `theta` encoding — the quantitative value that determines arc size
- `color` encoding — the categorical variable for segments

A **donut** chart is a pie with `innerRadius` > 0.

```python
mark_arc(innerRadius=60)  # donut
mark_arc(innerRadius=0)   # pie
```


In [ ]:
# Pie chart: count of cars by origin
pie_chart = alt.Chart(cars).mark_arc().encode(
    theta=alt.Theta('count()', title='Count'),  # arc size = count of rows
    color=alt.Color('Origin:N', title='Country of Origin'),
    tooltip=['Origin:N', 'count()']
).properties(
    title='Car Origin Distribution (pie)',
    width=250,
    height=250
)

# Donut chart: same data with innerRadius
donut_chart = alt.Chart(cars).mark_arc(innerRadius=70).encode(
    theta=alt.Theta('count()', title='Count'),
    color=alt.Color('Origin:N', title='Country of Origin'),
    tooltip=['Origin:N', 'count()']
).properties(
    title='Car Origin Distribution (donut)',
    width=250,
    height=250
)

# Display side by side using hconcat
pie_chart | donut_chart


### What just happened?
- `theta='count()'` maps the arc angle to row count — no pre-aggregation needed.
- `mark_arc(innerRadius=70)` creates a **donut** by cutting out the center.
- **`pie_chart | donut_chart`** uses Altair's `|` operator for horizontal concatenation — easy side-by-side comparison.
- Donut charts are generally preferred over pies: the center space can hold a summary label, and they're slightly easier to compare segment angles.
- **Tip**: avoid pie/donut charts for more than 5–6 categories — bar charts are easier to read at scale.


## Part 7 · `mark_text` — Labels on a bar chart

`mark_text` renders text at data coordinates. Combined with `mark_bar` in a layer,
you can add value labels directly on top of each bar.

Key `mark_text` properties:

| Property | Description |
|----------|------------|
| `align` | `'center'`, `'left'`, `'right'` |
| `baseline` | `'bottom'`, `'middle'`, `'top'` |
| `dy` | Vertical pixel offset from anchor |
| `fontSize` | Text size in pixels |


In [ ]:
# Bar chart with text labels showing the count on top of each bar
# Step 1: aggregate counts per origin
origin_counts = cars.groupby('Origin', as_index=False).size()
origin_counts.columns = ['Origin', 'Count']

base = alt.Chart(origin_counts)

bars = base.mark_bar(color='#4C78A8').encode(
    x=alt.X('Origin:N', title='Country of Origin', sort='-y'),
    y=alt.Y('Count:Q', title='Number of Models')
)

labels = base.mark_text(
    align='center',
    baseline='bottom',
    dy=-3,           # 3 px above the bar top
    fontSize=13,
    fontWeight='bold'
).encode(
    x=alt.X('Origin:N', sort='-y'),
    y=alt.Y('Count:Q'),
    text='Count:Q'   # the text content = the count value
)

# Layer bars + labels
labeled_bar = (bars + labels).properties(
    title='Car Models by Origin with Count Labels',
    width=320,
    height=260
)

labeled_bar


### What just happened?
- We built the bars and labels as **separate chart objects** sharing the same `base` data.
- `bars + labels` uses Altair's `+` operator for **layer composition** — both marks drawn on the same axes.
- `mark_text(dy=-3)` offsets the labels 3 pixels above the bar top; without this, the text would sit at the bar's top edge.
- `text='Count:Q'` maps the text content to the count value — without this, the mark has no text to show.
- **Sharing `base`** ensures both layers use the same data and encoding scale — no risk of misalignment.


## Part 8 · `mark_rule` — Reference line at mean value

`mark_rule` draws a horizontal or vertical line across the full chart extent.
It's perfect for adding a reference line at the mean, median, or any threshold.

To get the mean value inside the Vega-Lite spec:

```python
alt.Chart(df).mark_rule(color='red').encode(
    y='mean(column:Q)'
)
```

Altair computes the aggregate and draws the rule at that y value.


In [ ]:
# Scatter plot with a mean reference line for MPG
scatter = alt.Chart(cars).mark_point(opacity=0.5, size=30).encode(
    x=alt.X('Horsepower:Q', title='Horsepower'),
    y=alt.Y('Miles_per_Gallon:Q', title='MPG'),
    color='Origin:N'
)

mean_rule = alt.Chart(cars).mark_rule(
    color='crimson',
    strokeWidth=2,
    strokeDash=[6, 3]    # dashed line: 6px dash, 3px gap
).encode(
    y='mean(Miles_per_Gallon):Q',  # Altair computes the mean here
    tooltip=[alt.Tooltip('mean(Miles_per_Gallon):Q', format='.1f', title='Mean MPG')]
)

rule_chart = (scatter + mean_rule).properties(
    title='Horsepower vs MPG with Mean MPG Reference Line',
    width=500,
    height=320
)

rule_chart


### What just happened?
- `mark_rule` with `y='mean(Miles_per_Gallon):Q'` draws a **horizontal line** at the dataset mean MPG.
- `strokeDash=[6, 3]` makes the line dashed: 6px on, 3px off.
- `scatter + mean_rule` layers the two charts — same axes, different marks.
- **`mark_rule` can also be vertical**: use `x=` encoding instead of `y=` for a vertical reference.
- Use `mark_rule` for SLA thresholds, target lines, statistical baselines — any fixed or computed reference.


In [ ]:
# Challenge: Layered chart — mark_bar + mark_line + mark_text on one dataset
#
# Dataset: mean MPG per year from the cars dataset
# Goal: build a layered chart that shows:
#   1. mark_bar — mean MPG per year as bars
#   2. mark_line — trend line connecting mean MPG per year
#   3. mark_text — label each bar with the rounded mean MPG value
#
# Requirements:
#   - All three marks share the same base data (precomputed mean_per_year below)
#   - Bars should be light blue (#AEC8E2), line should be dark blue (#1565C0)
#   - Text labels: align='center', baseline='bottom', dy=-4, fontSize=10
#   - text encoding should show mean MPG rounded to 1 decimal (format='.1f')
#   - Set width=500, height=280 and a descriptive title
#
# Scaffold — wire together the three mark layers:

import altair as alt
from vega_datasets import data

cars = data.cars()

# Pre-aggregate: mean MPG per year
mean_per_year = (
    cars
    .dropna(subset=['Miles_per_Gallon', 'Year'])
    .groupby('Year', as_index=False)['Miles_per_Gallon']
    .mean()
    .rename(columns={'Miles_per_Gallon': 'Mean_MPG'})
)
mean_per_year['Mean_MPG'] = mean_per_year['Mean_MPG'].round(1)

# Your solution: define bars, line_layer, text_layer and combine with +
# base = alt.Chart(mean_per_year)
#
# bars = base.mark_bar(color='#AEC8E2').encode(
#     x=alt.X('Year:O', title='Year'),
#     y=alt.Y('Mean_MPG:Q', title='Mean MPG')
# )
#
# line_layer = base.mark_line(color='#1565C0', strokeWidth=2).encode(
#     x=...,
#     y=...
# )
#
# text_layer = base.mark_text(
#     align='center', baseline='bottom', dy=-4, fontSize=10
# ).encode(
#     x=...,
#     y=...,
#     text=alt.Text('Mean_MPG:Q', format='.1f')
# )
#
# challenge = (bars + line_layer + text_layer).properties(
#     title='...',
#     width=500,
#     height=280
# )
# challenge


---
## Day 3 key concepts recap

| Mark | Best for |
|------|----------|
| `mark_point` | Scatter plots; add jitter with `transform_calculate` |
| `mark_line` | Time series, trends; use `color` or `detail` for multi-series |
| `mark_bar` | Counts/aggregates; `xOffset` for grouped, default color for stacked |
| `mark_area` | Filled trends; `stack=None` + `opacity` for overlapping |
| `mark_rect` | Heatmaps; use a `domain` on the color scale for meaningful range |
| `mark_arc` | Pie/donut; `innerRadius>0` for donut; use for ≤5 categories |
| `mark_text` | Labels; layer over bars/points; use `dy` offset to avoid collision |
| `mark_rule` | Reference lines; `mean(col):Q` computes inline; add `strokeDash` |
| Layer operator `+` | Combine multiple marks on the same axes using a shared `base` |
| `hconcat` / `|` | Side-by-side charts |

> **Tip:** Every `mark_*` accepts `size`, `color`, `opacity` as direct kwargs for static (non-encoded) visual properties — e.g., `mark_point(size=100, opacity=0.6)`. Use `encode()` only for data-driven channels.

---
## What's next
**Day 4** → Encodings & Scales — deep dive into quantitative, ordinal, and temporal scales, custom color schemes, and binning.

Mark Day 3 complete in your [tracker](../index.html).
